# Financial RAG ingestion on Colab: PDFs → Qdrant `docs_text` + `visual_pages`

Runs [scripts/financial_ingest.py](../scripts/financial_ingest.py), the PDF part of
[docs/financial_rag_final_design.md](../docs/financial_rag_final_design.md) (sections 5 and 7).
Full guide: [docs/financial_ingestion_colab.md](../docs/financial_ingestion_colab.md).

| Collection · vector | What | Where |
|---|---|---|
| `docs_text` · `dense` + `bm25` | text and table chunks; number-aware sparse terms (`391,035` = `391035`) | RAM |
| `visual_pages` · `colpali` | full ColPali 1030 × 128, float16, no HNSW (rerank only) | disk (mmap) |
| `visual_pages` · `mean_rows`, `mean_cols` | 38 × 128 pooled, float16, HNSW (first stage) | RAM |
| `visual_pages` · `summary` | text vector of the page text | RAM |

**Runtime:** GPU. An L4 is preferred (bfloat16). On a T4 the default is float16; if ColPali returns non-finite
values, rerun with `--colpali-dtype float32`.

**Cheaper split (recommended):**
1. Run `STAGES = "text"` on a **CPU** runtime (page analysis, OCR, text vectors).
2. Then run `STAGES = "visual"` on a **GPU** runtime. The page-analysis cache travels with the snapshots, so
   the GPU only encodes pages.

**Data:** the download cell fetches the EDGAR corpus to Drive. It takes ~7 minutes and ~1.8 GB (**measured**),
and resumes if interrupted:
- the 100 ARS PDFs;
- the matching 10-K iXBRL HTML;
- the DEF 14A proxy statements;
- companyfacts and submissions JSON.

**Rules:**
- Qdrant storage stays on `/content` (local disk), never on the Drive mount.
- PDFs, snapshots and the ingest state live on Drive.
- Everything is resumable. After a disconnect, rerun cells 1–5 and then the ingest cell; finished filings
  are skipped.

In [ ]:
# 1. Settings
DRIVE_ROOT = "/content/drive/MyDrive/capstone_rag"     # persistent: EDGAR corpus, snapshots, ingest state
WORK = "/content/rag"                                   # fast local disk: Qdrant storage and caches
REPO = "https://github.com/rama159065/Capstone.git"     # private repo: see the note in cell 3
BRANCH = "capstone_multimodal_rag"
SEC_USER_AGENT = "capstone-research your.name@example.com"   # SEC requires a contact e-mail
STAGES = "text,visual"    # or "text" on a CPU runtime, then "visual" on a GPU runtime

In [ ]:
# 2. Drive, folders, GPU
import os
from google.colab import drive
drive.mount("/content/drive")
for path in (f"{DRIVE_ROOT}/edgar/pdfs", f"{DRIVE_ROOT}/snapshots", WORK):
    os.makedirs(path, exist_ok=True)
os.environ["RAG_INGEST_DIR"] = WORK          # --root default for every command below
os.environ["SEC_USER_AGENT"] = SEC_USER_AGENT
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "no GPU: run only the text stage"
!df -h /content | tail -1

In [ ]:
# 3. Code and dependencies.
# Private repo: store a GitHub token as the Colab secret GITHUB_TOKEN, then
#   from google.colab import userdata
#   REPO = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/rama159065/Capstone.git"
!test -d /content/Capstone || git clone --depth 1 -b {BRANCH} {REPO} /content/Capstone
%cd /content/Capstone
!pip -q install "qdrant-client>=1.12,<2" "pymupdf>=1.24,<2" "transformers>=4.57,<5" "sentence-transformers>=3" "rapidocr>=3,<4" onnxruntime
# Optional GPU OCR: pip uninstall -y onnxruntime && pip install onnxruntime-gpu, then add --ocr-cuda

In [ ]:
# 4. Qdrant server on local disk, bound to localhost only
import subprocess, time, urllib.request
QDRANT_DIR = "/content/qdrant"
if not os.path.exists(f"{QDRANT_DIR}/qdrant"):
    os.makedirs(QDRANT_DIR, exist_ok=True)
    # For the final runs, pin a tested version: .../releases/download/vX.Y.Z/qdrant-x86_64-unknown-linux-gnu.tar.gz
    !curl -sL https://github.com/qdrant/qdrant/releases/latest/download/qdrant-x86_64-unknown-linux-gnu.tar.gz | tar xz -C {QDRANT_DIR}
env = {**os.environ, "QDRANT__SERVICE__HOST": "127.0.0.1",
       "QDRANT__STORAGE__STORAGE_PATH": f"{WORK}/qdrant",
       "QDRANT__STORAGE__SNAPSHOTS_PATH": f"{WORK}/snapshots",   # = <root>/snapshots, used by restore
       "QDRANT__TELEMETRY_DISABLED": "true"}
try:
    urllib.request.urlopen("http://127.0.0.1:6333/readyz", timeout=2)
    print("Qdrant already running")
except Exception:
    qdrant = subprocess.Popen([f"{QDRANT_DIR}/qdrant"], cwd=QDRANT_DIR, env=env,
                              stdout=open(f"{WORK}/qdrant.log", "ab"), stderr=subprocess.STDOUT)
    for _ in range(60):
        try:
            urllib.request.urlopen("http://127.0.0.1:6333/readyz", timeout=2)
            break
        except Exception:
            time.sleep(1)
print(urllib.request.urlopen("http://127.0.0.1:6333").read().decode())
os.environ["QDRANT_URL"] = "http://127.0.0.1:6333"

In [ ]:
# 5. Restore the previous session's snapshots and ingest state (nothing to do on the first session)
import glob
if glob.glob(f"{DRIVE_ROOT}/snapshots/*.snapshot") and not os.path.exists(f"{WORK}/ingest.sqlite3"):
    !python -m scripts.financial_ingest restore --snapshot-dir {DRIVE_ROOT}/snapshots
else:
    print("nothing to restore (first session, or this runtime already has the index)")

## Download the corpus (EDGAR → Drive)

Fetched from SEC EDGAR, one row per filing in `corpus_manifest.jsonl` (linked by `filing_id`):

| Folder | Content |
|---|---|
| `pdfs/` | ARS PDFs, SHA-256 checked against the manifest |
| `10k/` | 10-K iXBRL HTML, plus sibling iXBRL documents when the statements are split out (Wells Fargo) |
| `proxy/` | DEF 14A proxy statements |
| `companyfacts/`, `submissions/` | SEC JSON, once per company |

Rerunning it fetches only what is missing.

In [ ]:
!python -m scripts.download_edgar_corpus --output-dir {DRIVE_ROOT}/edgar --user-agent "{SEC_USER_AGENT}"
!du -sh {DRIVE_ROOT}/edgar/*

## Smoke test (embedded Qdrant, separate folder, one filing, 8 pages)

Checks the download, page analysis, OCR, ColPali model and pooling before the long run. It does not touch the
server collections.

In [ ]:
!QDRANT_URL= python -m scripts.financial_ingest ingest --root /content/rag_smoke --stages {STAGES} \
    --pdf-dir {DRIVE_ROOT}/edgar/pdfs --download-missing --limit-filings 1 --max-pages-per-pdf 8 --reset 2>&1 | grep -v pymupdf_layout
!QDRANT_URL= python -m scripts.financial_ingest status --root /content/rag_smoke

## Full ingest

Snapshots go to Drive after every batch of 10 filings. Each is checksum-verified, and only the newest is kept.
Progress is one JSON line per filing and stage.

**For 100 filings (14.7k pages):**
- **Text stage:** ~0.5 s/page/core (**measured** locally), so ~1 h on 2 workers.
- **Visual stage:** ~40% of pages. **Measured:** 24–27% for clean filings, up to 95% for a garbled one.
  That is ~5,900 pages at ~0.5–1.5 pages/s on a T4 (**estimate**), so ~1–3.5 GPU-hours.

In [ ]:
!python -m scripts.financial_ingest ingest --stages {STAGES} --pdf-dir {DRIVE_ROOT}/edgar/pdfs --download-missing \
    --snapshot-dir {DRIVE_ROOT}/snapshots --batch-size 10 --workers 2 --colpali-batch 4 2>&1 | grep -v pymupdf_layout

In [ ]:
# Status: points per collection, filings per stage, failures (rerun the ingest cell to retry them)
!python -m scripts.financial_ingest status
!du -sh {WORK}/qdrant {DRIVE_ROOT}/snapshots

In [ ]:
# Retrieval checks: exact-number text search, then the visual path (pooled prefetch -> full MaxSim rerank)
!python -m scripts.financial_ingest query "total revenue 130,497" --ticker NVDA
!python -m scripts.financial_ingest query "revenue by market platform bar chart" --ticker NVDA --visual

## Next

- **Serving:** see section 11.1 of the design. Start Qdrant on the same `WORK` folder (cells 4–5), then FastAPI
  and Streamlit.
- **Facts DB:** this notebook does not write the 10-K HTML / XBRL facts (store A); that is a separate step.